In [ ]:
""" Import libraries """

import LoadSaveFunctions as lsf
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load Dataset

data = lsf.load_excel_file()

print(data.head(5))

In [ ]:
def bivariate_fit(xi, yi, dxi, dyi, ri=0.0, b0=1.0, maxIter=1e6):
    """Make a linear bivariate fit to xi, yi data using York et al. (2004).
    This is an implementation of the line fitting algorithm presented in:
    York, D et al., Unified equations for the slope, intercept, and standard
    errors of the best straight line, American Journal of Physics, 2004, 72,
    3, 367-375, doi = 10.1119/1.1632486
    See especially Section III and Table I. The enumerated steps below are
    citations to Section III
    Parameters:
      xi, yi      x and y data points
      dxi, dyi    errors for the data points xi, yi
      ri          correlation coefficient for the weights
      b0          initial guess b
      maxIter     float, maximum allowed number of iterations
    Returns:
      a           y-intercept, y = a + bx
      b           slope
      S           goodness-of-fit estimate
      sigma_a     standard error of a
      sigma_b     standard error of b
      cov_matrix  covariance matrix of a and b
    """
    b = b0
    wxi = 1.0 / dxi**2.0
    wyi = 1.0 / dyi**2.0
    alphai = (wxi * wyi)**0.5
    b_diff = 999.0
    tol = 1.0e-8
    iIter = 1

    while (abs(b_diff) >= tol) & (iIter <= maxIter):
        b_prev = b
        Wi = (wxi * wyi) / (wxi + b**2.0 * wyi - 2.0*b*ri*alphai)
        x_bar = np.sum(Wi * xi) / np.sum(Wi)
        y_bar = np.sum(Wi * yi) / np.sum(Wi)
        Ui = xi - x_bar
        Vi = yi - y_bar
        betai = Wi * (Ui / wyi + b*Vi / wxi - (b*Ui + Vi) * ri / alphai)
        b = np.sum(Wi * betai * Vi) / np.sum(Wi * betai * Ui)
        b_diff = b - b_prev
        iIter += 1

    a = y_bar - b * x_bar
    S = np.sum(Wi * (yi - b*xi - a)**2.0)
    xi_adj = x_bar + betai
    xi_adj_bar = np.sum(Wi * xi_adj) / np.sum(Wi)
    ui = xi_adj - xi_adj_bar
    sigma_b = np.sqrt(1.0 / np.sum(Wi * ui**2))
    sigma_a = np.sqrt(1.0 / np.sum(Wi) + xi_adj_bar**2 * sigma_b**2)
    cov = -xi_adj_bar * sigma_b**2
    cov_matrix = np.array([[sigma_b**2, cov], [cov, sigma_a**2]])
    
    if iIter <= maxIter:
        return a, b, S, cov_matrix, sigma_a, sigma_b
    else:
        print("bivariate_fit.py exceeded maximum number of iterations, maxIter = {:}".format(maxIter))
        return np.nan, np.nan, np.nan, np.nan

def prediction_interval(xi, yi, dxi, dyi, x_new, ri=0.0, confidence=0.95):
    """Calculates the prediction error and confidence interval for a new x value."""
    from scipy.stats import t
    a, b, S, cov_matrix, sigma_a, sigma_b = bivariate_fit(xi, yi, dxi, dyi, ri)
    sigma_b, sigma_a = np.sqrt(cov_matrix[0, 0]), np.sqrt(cov_matrix[1, 1])
    n = len(xi)
    t_value = t.ppf((1 + confidence) / 2, df=n-2)
    
    y_new = a + b * x_new
    sigma_y_new = np.sqrt(S / (n - 2) + (x_new**2 * sigma_b**2) + sigma_a**2)
    
    margin_error = t_value * sigma_y_new
    lower_bound = y_new - margin_error
    upper_bound = y_new + margin_error
    
    return y_new, sigma_y_new, lower_bound, upper_bound


In [ ]:
[a_bivar, b_bivar, S, cov, sigma_a, sigma_b] = bivariate_fit(data['d18Op'], data['d18Ow'], data['d18OpSD'], data['d18OwSD'])

if a_bivar < 0:
    label_york = 'y={:1.2f}±{:1.2f}x{:1.2f}±{:1.2f}'.format(b_bivar, sigma_b, a_bivar, sigma_a)
else:
    label_york = 'y={:1.2f}±{:1.2f}x+{:1.2f}±{:1.2f}'.format(b_bivar, sigma_b, a_bivar, sigma_a)
print("York regression: " + label_york)

In [ ]:
x_new = 25.4
y_pred, sigma_y_pred, lower_bound, upper_bound = prediction_interval(data['d18Op'], data['d18Ow'], data['d18OpSD'], data['d18OwSD'], x_new)
print(f"For x = {x_new:.3f}, Predicted y: {y_pred:.3f} ± {sigma_y_pred:.3f} ; Lower bound: {lower_bound:.3f}, Upper bound: {upper_bound:.3f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t

def bivariate_fit(xi, yi, dxi, dyi, ri=0.0, b0=1.0, maxIter=1e6):
    """York et al. (2004) regression with confidence and prediction intervals."""
    b = b0
    wxi = 1.0 / dxi**2.0
    wyi = 1.0 / dyi**2.0
    alphai = (wxi * wyi)**0.5
    b_diff = 999.0
    tol = 1.0e-8
    iIter = 1

    while (abs(b_diff) >= tol) & (iIter <= maxIter):
        b_prev = b
        Wi = (wxi * wyi) / (wxi + b**2.0 * wyi - 2.0*b*ri*alphai)
        x_bar = np.sum(Wi * xi) / np.sum(Wi)
        y_bar = np.sum(Wi * yi) / np.sum(Wi)
        Ui = xi - x_bar
        Vi = yi - y_bar
        betai = Wi * (Ui / wyi + b*Vi / wxi - (b*Ui + Vi) * ri / alphai)
        b = np.sum(Wi * betai * Vi) / np.sum(Wi * betai * Ui)
        b_diff = b - b_prev
        iIter += 1

    a = y_bar - b * x_bar
    S = np.sum(Wi * (yi - b*xi - a)**2.0)
    xi_adj = x_bar + betai
    xi_adj_bar = np.sum(Wi * xi_adj) / np.sum(Wi)
    ui = xi_adj - xi_adj_bar
    sigma_b = np.sqrt(1.0 / np.sum(Wi * ui**2))
    sigma_a = np.sqrt(1.0 / np.sum(Wi) + xi_adj_bar**2 * sigma_b**2)
    cov = -xi_adj_bar * sigma_b**2
    cov_matrix = np.array([[sigma_b**2, cov], [cov, sigma_a**2]])

    return a, b, S, cov_matrix, Wi, sigma_a, sigma_b

def compute_intervals(x, a, b, cov_matrix, Wi, confidence=0.95):
    """Compute confidence and prediction intervals."""
    n = len(x)
    x_mean = np.mean(x)
    t_value = 2.0  # Approximate 95% t-distribution value
    sigma_b2, cov_ab, sigma_a2 = cov_matrix[0, 0], cov_matrix[0, 1], cov_matrix[1, 1]
    y_pred = a + b * x
    se_fit = np.sqrt(sigma_a2 + 2 * cov_ab * x + sigma_b2 * x**2)
    ci = t_value * se_fit  # Confidence Interval
    pi = t_value * np.sqrt(se_fit**2 + (1 / np.sum(Wi)))  # Prediction Interval
    return y_pred, ci, pi

def prediction_interval(x, a, b, S, cov_matrix, sigma_a, sigma_b, confidence=0.95):
    """Calculates the prediction error and confidence interval for a new x value."""
    n = len(x)
    sigma_b, sigma_a = np.sqrt(cov_matrix[0, 0]), np.sqrt(cov_matrix[1, 1])
    t_value = t.ppf((1 + confidence) / 2, df=n-2)
    
    y_pred = a + b * x
    sigma_y_pred = np.sqrt(S / (n - 2) + (x**2 * sigma_b**2) + sigma_a**2)
    
    margin_error = t_value * sigma_y_pred
    lower_bound = y_pred - margin_error
    upper_bound = y_pred + margin_error
    
    return lower_bound, upper_bound



def plot_regression(x, y, dxi, dyi, ri=0.0, b0=1.0):
    """Plot York regression with confidence and prediction intervals."""
    a, b, S, cov_matrix, Wi, sigma_a, sigma_b = bivariate_fit(x, y, dxi, dyi, ri, b0)
    if a < 0:
            label_york = 'York regression: y={:1.2f}±{:1.2f}x{:1.2f}±{:1.2f}'.format(b, sigma_b, a, sigma_a)
    else:
        label_york = 'York regression: y={:1.2f}±{:1.2f}x+{:1.2f}±{:1.2f}'.format(b, sigma_b, a, sigma_a)

    x_fit = np.linspace(min(x), max(x), 100)
    y_fit, ci, pi = compute_intervals(x_fit, a, b, cov_matrix, Wi)
    lower_bound, upper_bound = prediction_interval(x_fit, a, b, S, cov_matrix, sigma_a, sigma_b, confidence=0.95)
    
    plt.figure(figsize=(8, 6))
    plt.errorbar(x, y, xerr=dxi, yerr=dyi, fmt='o', label='Data', capsize=3)
    plt.plot(x_fit, y_fit, 'r-', label=label_york)
    plt.fill_between(x_fit, y_fit - ci, y_fit + ci, color='blue', alpha=0.2, label='Confidence Interval')
    # plt.fill_between(x_fit, y_fit - pi, y_fit + pi, color='orange', alpha=0.2, label='Prediction Interval')
    plt.fill_between(x_fit, lower_bound, upper_bound, color='orange', alpha=0.2, label='Prediction Interval')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.legend()
    plt.title('York Regression with Confidence & Prediction Intervals')
    plt.show()

# Example usage:
x = np.array([1, 2, 3, 4, 5])
# y = np.array([2.1, 2.9, 3.8, 5.1, 5.9])
y = np.array([2.1, 3.2, 4.1, 4, 5.9])
dxi = np.array([0.1, 0.1, 0.1, 0.1, 0.1])
dyi = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
plot_regression(x, y, dxi, dyi)

In [ ]:
plot_regression(data['d18Op'], data['d18Ow'], data['d18OpSD'], data['d18OwSD'])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t

def bivariate_fit(xi, yi, dxi, dyi, ri=0.0, b0=1.0, maxIter=1e6):
    """York et al. (2004) regression with confidence and prediction intervals."""
    b = b0
    wxi = 1.0 / dxi**2.0
    wyi = 1.0 / dyi**2.0
    alphai = (wxi * wyi)**0.5
    b_diff = 999.0
    tol = 1.0e-8
    iIter = 1

    while (abs(b_diff) >= tol) & (iIter <= maxIter):
        b_prev = b
        Wi = (wxi * wyi) / (wxi + b**2.0 * wyi - 2.0*b*ri*alphai)
        x_bar = np.sum(Wi * xi) / np.sum(Wi)
        y_bar = np.sum(Wi * yi) / np.sum(Wi)
        Ui = xi - x_bar
        Vi = yi - y_bar
        betai = Wi * (Ui / wyi + b*Vi / wxi - (b*Ui + Vi) * ri / alphai)
        b = np.sum(Wi * betai * Vi) / np.sum(Wi * betai * Ui)
        b_diff = b - b_prev
        iIter += 1

    a = y_bar - b * x_bar
    S = np.sum(Wi * (yi - b*xi - a)**2.0)
    xi_adj = x_bar + betai
    xi_adj_bar = np.sum(Wi * xi_adj) / np.sum(Wi)
    ui = xi_adj - xi_adj_bar
    sigma_b = np.sqrt(1.0 / np.sum(Wi * ui**2))
    sigma_a = np.sqrt(1.0 / np.sum(Wi) + xi_adj_bar**2 * sigma_b**2)
    cov = -xi_adj_bar * sigma_b**2
    cov_matrix = np.array([[sigma_b**2, cov], [cov, sigma_a**2]])

    return a, b, S, cov_matrix, Wi, sigma_a, sigma_b

def confidence_interval(x, a, b, cov_matrix):
    """Compute confidence interval."""
    n = len(x)
    x_mean = np.mean(x)
    t_value = 2.0  # Approximate 95% t-distribution value
    sigma_b2, cov_ab, sigma_a2 = cov_matrix[0, 0], cov_matrix[0, 1], cov_matrix[1, 1]
    y_pred = a + b * x
    se_fit = np.sqrt(sigma_a2 + 2 * cov_ab * x + sigma_b2 * x**2)
    ci = t_value * se_fit  # Confidence Interval
    return y_pred, ci

def prediction_interval(x, x_data, a, b, S, cov_matrix, confidence=0.95):
    """Computes the prediction interval for a given x using York regression parameters.

    Parameters:
    - x: The x value(s) for which to compute the interval.
    - x_data: The dataset of x values.
    - S: Residual sum of squares from the regression.
    - cov_matrix: Covariance matrix of (b, a) from York regression.
    - confidence: Confidence level (default: 0.95).

    Returns:
    - margin_error: The margin of error (prediction envelope) for the x value(s).
    """

    # Calculate necessary statistics
    n = len(x_data)
    x_mean = np.mean(x_data)
    sum_x_squared = np.sum((x_data - x_mean)**2)

    # Extract standard deviations and covariance from the covariance matrix
    sigma_b = np.sqrt(cov_matrix[0, 0])  # Std dev of slope
    sigma_a = np.sqrt(cov_matrix[1, 1])  # Std dev of intercept
    cov_ab = cov_matrix[0, 1]  # Covariance between slope and intercept

    # Compute t-critical value
    t_value = stats.t.ppf((1 + confidence) / 2, df=n-2)

    # Compute standard error of prediction
    SE_pred = np.sqrt(
        S / (n - 2) * (1 + 1/n + (x - x_mean)**2 / sum_x_squared) +
        x**2 * sigma_b**2 +
        sigma_a**2 +
        2 * x * cov_ab  # Covariance term
    )

    # Compute margin of error
    y_pred = a + b * x
    margin_error = t_value * SE_pred

    lower_bound = y_pred - margin_error
    upper_bound = y_pred + margin_error

    return lower_bound, upper_bound


def plot_regression(x, y, dxi, dyi, ri=0.0, b0=1.0):
    """Plot York regression with confidence and prediction intervals."""
    a, b, S, cov_matrix, Wi, sigma_a, sigma_b = bivariate_fit(x, y, dxi, dyi, ri, b0)
    if a < 0:
            label_york = 'York regression: y={:1.2f}±{:1.2f}x{:1.2f}±{:1.2f}'.format(b, sigma_b, a, sigma_a)
    else:
        label_york = 'York regression: y={:1.2f}±{:1.2f}x+{:1.2f}±{:1.2f}'.format(b, sigma_b, a, sigma_a)

    x_fit = np.linspace(min(x), max(x), 100)    #create a range of datapoints to create the enveloppes

    y_fit, ci = confidence_interval(x_fit, a, b, cov_matrix)
    
    lower_bound, upper_bound = prediction_interval(x_fit, x, a, b, S, cov_matrix, confidence=0.95)
    
    plt.figure(figsize=(8, 6))
    plt.errorbar(x, y, xerr=dxi, yerr=dyi, fmt='o', label='Data', capsize=3)
    plt.plot(x_fit, y_fit, 'r-', label=label_york)
    plt.fill_between(x_fit, y_fit - ci, y_fit + ci, color='blue', alpha=0.2, label='Confidence Interval')
    # plt.fill_between(x_fit, y_fit - pi, y_fit + pi, color='orange', alpha=0.2, label='Prediction Interval')
    plt.fill_between(x_fit, lower_bound, upper_bound, color='orange', alpha=0.2, label='Prediction Interval')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.legend()
    plt.title('York Regression with Confidence & Prediction Intervals')
    plt.show()


In [ ]:
plot_regression(data['d18Op'], data['d18Ow'], data['d18OpSD'], data['d18OwSD'])

In [ ]:
x = np.array([1, 2, 3, 4, 5])
y = np.array([2.1, 2.9, 3.8, 5.1, 5.9])
dxi = np.array([0.1, 0.1, 0.1, 0.1, 0.1])
dyi = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
plot_regression(x, y, dxi, dyi)